# Hybrid Recommender — In-Memory (no graph DB)

Same 2-hop algorithm as the Neo4j version, but with FAISS + a sparse CSR ratings matrix. No DB, no caps, runs in milliseconds, handles all 309k users.

**Algorithm**
1. Use the existing user FAISS index → top-K similar users (cosine score).
2. Look up those users' ratings in a sparse `users × animes` matrix.
3. Score each candidate anime as `sum(user_similarity * rating/10)`, mask animes the target already rated, return top-K.

This is mathematically identical to the Cypher query in `hybrid.ipynb` — just one sparse matmul instead of a graph traversal.

## 1. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from scipy.sparse import csr_matrix

DATA_DIR     = Path.cwd().parent / "data"
ARTIFACT_DIR = Path.cwd().parent / "faiss_artifacts"

SIMILAR_USERS_K = 50
TOP_K           = 10

## 2. Load artifacts

In [2]:
# Anime FAISS (for content vectors) + MAL_ID ↔ anime_faiss_id
with open(ARTIFACT_DIR / "mal_id_to_faiss_id.json", "r", encoding="utf-8") as f:
    mal_id_to_aidx = {int(k): int(v) for k, v in json.load(f).items()}
aidx_to_mal_id = {v: k for k, v in mal_id_to_aidx.items()}
N_ANIME = len(mal_id_to_aidx)

# User FAISS + user_id ↔ user_faiss_id
user_index = faiss.read_index(str(ARTIFACT_DIR / "user_faiss_index.bin"))
with open(ARTIFACT_DIR / "user_id_to_faiss_id.json", "r", encoding="utf-8") as f:
    user_id_to_uidx = {int(k): int(v) for k, v in json.load(f).items()}
uidx_to_user_id = {v: k for k, v in user_id_to_uidx.items()}
N_USER = len(user_id_to_uidx)

print(f"{N_USER} users, {N_ANIME} animes")

309484 users, 16206 animes


In [3]:
# Titles for nice output
anime_titles = (
    pd.read_csv(DATA_DIR / "anime.csv", usecols=["MAL_ID", "Name"])
    .set_index("MAL_ID")["Name"]
    .to_dict()
)

## 3. Build sparse ratings matrix

`ratings_csr[uidx, aidx] = rating`. Rows in user-FAISS-id order, cols in anime-FAISS-id order, so `user_index` results plug straight in.

In [4]:
ratings_df = pd.read_csv(DATA_DIR / "rating_complete.csv")
ratings_df = ratings_df[ratings_df["rating"] > 0]
ratings_df = ratings_df[ratings_df["anime_id"].isin(mal_id_to_aidx)]
ratings_df = ratings_df[ratings_df["user_id"].isin(user_id_to_uidx)]

rows = ratings_df["user_id"].map(user_id_to_uidx).to_numpy()
cols = ratings_df["anime_id"].map(mal_id_to_aidx).to_numpy()
data = ratings_df["rating"].to_numpy(dtype="float32") / 10.0   # normalize to [0,1]

ratings_csr = csr_matrix((data, (rows, cols)), shape=(N_USER, N_ANIME))
print(f"ratings matrix: {ratings_csr.shape}, nnz={ratings_csr.nnz:,}")
del ratings_df, rows, cols, data

ratings matrix: (309484, 16206), nnz=56,726,814


## 4. Recommend

In [5]:
def recommend(user_id: int, top_k: int = TOP_K, k_users: int = SIMILAR_USERS_K) -> pd.DataFrame:
    user_id = int(user_id)
    if user_id not in user_id_to_uidx:
        raise KeyError(f"user {user_id} not in index")

    uidx = user_id_to_uidx[user_id]
    query_vec = user_index.reconstruct(uidx).reshape(1, -1)

    # 1st hop: similar users (ask for k+1 because the target itself comes back)
    sims, peers = user_index.search(query_vec, k_users + 1)
    mask = peers[0] != uidx
    peer_idx = peers[0][mask][:k_users]
    peer_sim = sims[0][mask][:k_users].astype("float32")

    # 2nd hop: score = peer_sim @ ratings_csr[peer_idx]  -> shape (N_ANIME,)
    scores = peer_sim @ ratings_csr[peer_idx]
    scores = np.asarray(scores).ravel()

    # support = how many peers actually rated each anime
    support = np.asarray((ratings_csr[peer_idx] > 0).sum(axis=0)).ravel()

    # mask animes the target already rated
    seen = ratings_csr.getrow(uidx).indices
    scores[seen] = -np.inf

    # top-k
    top = np.argpartition(-scores, top_k)[:top_k]
    top = top[np.argsort(-scores[top])]

    return pd.DataFrame({
        "mal_id":  [aidx_to_mal_id[i] for i in top],
        "title":   [anime_titles.get(aidx_to_mal_id[i], str(aidx_to_mal_id[i])) for i in top],
        "score":   scores[top],
        "support": support[top],
    })

In [6]:
demo_user = next(iter(user_id_to_uidx))
print(f"recommendations for user {demo_user}:")
recommend(demo_user, top_k=10)

recommendations for user 0:


,mal_id,title,score,support
0,853,Ouran Koukou Host Club,34.474644,43
1,1535,Death Note,34.127110,42
2,523,Tonari no Totoro,34.043434,42
3,512,Majo no Takkyuubin,30.898603,40
4,20,Naruto,30.801006,41
5,585,Mimi wo Sumaseba,29.528889,37
6,120,Fruits Basket,28.340931,36
7,597,Neko no Ongaeshi,27.683968,36
8,513,Tenkuu no Shiro Laputa,27.229523,35
9,572,Kaze no Tani no Nausicaä,26.124693,33


In [7]:
# What they already liked
uidx = user_id_to_uidx[demo_user]
row  = ratings_csr.getrow(uidx)
seen = pd.DataFrame({
    "title":  [anime_titles.get(aidx_to_mal_id[i], str(aidx_to_mal_id[i])) for i in row.indices],
    "rating": (row.data * 10).astype(int),
}).sort_values("rating", ascending=False).head(10).reset_index(drop=True)
seen

,title,rating
0,Toki wo Kakeru Shoujo,10
1,Hotaru no Haka,10
2,Tonari no Yamada-kun,10
3,Ghost Hunt,10
4,Igano Kabamaru,9
5,Fullmetal Alchemist,9
6,Jungle no Ouja Taa-chan,9
7,One Piece Movie 1,9
8,Fate/stay night,9
9,Fullmetal Alchemist: The Conqueror of Shamballa,9


In [8]:
# Quick timing
%timeit recommend(demo_user)

455 μs ± 6.38 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
